# Experiment 04: Сравнение моделей (MLP)
## Цель эксперимента
1. Обучить MLP (нейронную сеть) на тех же данных
2. Сравнить с XGBoost из предыдущего эксперимента
3. Залогировать всё в MLflow для наблюдаемости

## Используемые данные
- Датасет: Churn_Modelling.csv (10,000 клиентов)
- Признаки: 13 (после feature engineering)
- Целевая переменная: Exited (отток)

In [1]:
import sys
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import joblib
from datetime import datetime

# Настройки
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-darkgrid')

# Добавляем путь к src
sys.path.append('..')
sys.path.append(os.path.abspath('..'))

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, roc_curve,
    accuracy_score, precision_score, recall_score, f1_score
)
from sklearn.neural_network import MLPClassifier
import xgboost as xgb

# Импортируем наш препроцессор
from src.data.preprocess import DataPreprocessor

print("Библиотеки загружены")


Библиотеки загружены


### Загрузка конфигурации

In [2]:
config_path = "../configs/config.yaml"
if os.path.exists(config_path):
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    print("Конфигурация загружена из config.yaml")
else:
    config = {
        'random_state': 42,
        'test_size': 0.2
    }
    print("Используется конфигурация по умолчанию")

RANDOM_STATE = config.get('random_state', 42)
TEST_SIZE = config.get('test_size', 0.2)

print(f"Random state: {RANDOM_STATE}")
print(f"Test size: {TEST_SIZE}")

Конфигурация загружена из config.yaml
Random state: 42
Test size: 0.2


In [8]:
import mlflow
mlflow.set_experiment("Bank_Churn")
mlflow.set_tracking_uri("http://localhost:5000")

2026/05/31 02:25:29 INFO mlflow.tracking.fluent: Experiment with name 'Bank_Churn' does not exist. Creating a new experiment.


In [6]:
# Загружаем сырые данные
df_raw = pd.read_csv('../data/raw/Churn_Modelling.csv')
print(f"Загружено {len(df_raw)} записей")
print(f"Исходные колонки: {list(df_raw.columns)}")

# Инициализируем препроцессор
preprocessor = DataPreprocessor()

# Обрабатываем данные (fit_scaler=True для обучения)
X_scaled, y, df_processed = preprocessor.preprocess(df_raw, fit_scaler=True)

# Сохраняем препроцессор для сервиса
os.makedirs('../artifacts', exist_ok=True)
preprocessor.save('../artifacts/preprocessor.pkl')

target_col = y.name if hasattr(y, 'name') else 'Exited'

# Признаки - это все колонки, кроме целевой
feature_names = [col for col in df_processed.columns if col != target_col]
print(f"Имена признаков: {feature_names[:5]}... (всего {len(feature_names)})")
print(f"\nПосле предобработки:")
print(f"  - Признаков: {X_scaled.shape[1]}")
print(f"  - Целевая переменная: {target_col}")
print(f"  - Распределение: {y.value_counts().to_dict()}")

INFO:src.data.preprocess:DataPreprocessor инициализирован
INFO:src.data.preprocess:Создано 21 признаков
INFO:src.data.preprocess:Данные очищены. Форма: (10000, 14)
INFO:src.data.preprocess:Scaler обучен и применён
INFO:src.data.preprocess:Препроцессор сохранён в ../artifacts/preprocessor.pkl


Загружено 10000 записей
Исходные колонки: ['RowNumber', 'CustomerId', 'Surname', 'CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited']
Имена признаков: ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts']... (всего 13)

После предобработки:
  - Признаков: 13
  - Целевая переменная: Exited
  - Распределение: {0: 7963, 1: 2037}


In [7]:
# Разделение на train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, 
    test_size=TEST_SIZE, 
    random_state=RANDOM_STATE,
    stratify=y
)

print(f"\nРазделение данных:")
print(f"  - Train: {X_train.shape[0]} записей (отток: {y_train.mean():.2%})")
print(f"  - Test: {X_test.shape[0]} записей (отток: {y_test.mean():.2%})")


Разделение данных:
  - Train: 8000 записей (отток: 20.38%)
  - Test: 2000 записей (отток: 20.35%)


### MLP

In [18]:
from sklearn.utils.class_weight import compute_sample_weight

# Рассчитываем веса для каждой выборки
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

print(f"Вес для класса 0: {sample_weights[y_train == 0][0]:.3f}")
print(f"Вес для класса 1: {sample_weights[y_train == 1][0]:.3f}")

# Обучаем с sample_weight
mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32, 16),
    activation='relu',
    solver='adam',
    alpha=0.001,
    batch_size=32,
    learning_rate='adaptive',
    learning_rate_init=0.001,
    max_iter=300,
    early_stopping=True,
    validation_fraction=0.1,
    random_state=42
)

Вес для класса 0: 0.628
Вес для класса 1: 2.454


In [19]:
# Логируем в MLflow
with mlflow.start_run(run_name="Bank_Churn_MLP", nested=True):
    # Логируем параметры
    mlflow.log_params(mlp_params)
    
    # Обучаем
    print("Обучение MLP")
    mlp.fit(X_train, y_train, sample_weight=sample_weights)
    
    # Предсказания
    y_pred_mlp = mlp.predict(X_test)
    y_pred_proba_mlp = mlp.predict_proba(X_test)[:, 1]
    
    # Метрики
    acc_mlp = accuracy_score(y_test, y_pred_mlp)
    prec_mlp = precision_score(y_test, y_pred_mlp)
    rec_mlp = recall_score(y_test, y_pred_mlp)
    f1_mlp = f1_score(y_test, y_pred_mlp)
    roc_mlp = roc_auc_score(y_test, y_pred_proba_mlp)
    
    # Логируем метрики
    mlp_metrics = {
        "accuracy": acc_mlp,
        "precision": prec_mlp,
        "recall": rec_mlp,
        "f1": f1_mlp,
        "roc_auc": roc_mlp
    }
    mlflow.log_metrics(mlp_metrics)
    
    # Логируем количество итераций
    mlflow.log_metric("mlp_n_iter", mlp.n_iter_)
    
    # Сохраняем модель
    os.makedirs('../artifacts/models', exist_ok=True)
    mlp_path = '../artifacts/models/mlp_model.pkl'
    joblib.dump(mlp, mlp_path)
    mlflow.log_artifact(mlp_path)
    
    print(f"\nMLP результаты:")
    print(f"  - Accuracy:  {acc_mlp:.4f}")
    print(f"  - Precision: {prec_mlp:.4f}")
    print(f"  - Recall:    {rec_mlp:.4f}")
    print(f"  - F1-Score:  {f1_mlp:.4f}")
    print(f"  - ROC-AUC:   {roc_mlp:.4f}")
    print(f"  - Итераций:  {mlp.n_iter_}")

Обучение MLP

MLP результаты:
  - Accuracy:  0.7675
  - Precision: 0.4575
  - Recall:    0.7666
  - F1-Score:  0.5730
  - ROC-AUC:   0.8555
  - Итераций:  13
🏃 View run Bank_Churn_MLP at: http://localhost:5000/#/experiments/1/runs/04dc00bacd734d8d8fcbed350edf9c0b
🧪 View experiment at: http://localhost:5000/#/experiments/1
